In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# ⚖️ Legal Clause Retriever

## What We're Building

A tool that searches through legal contracts and:
1. **Retrieves** specific clauses by topic (termination, liability, IP, etc.)
2. **Identifies risky clauses** — unfavorable or unusual terms
3. **Flags missing sections** — important clauses that should be there but aren't

## Key Concept: Hybrid Retrieval

Legal text is tricky — it uses precise terminology. Pure semantic search may miss
a clause that uses the exact legal term you're looking for.

**Hybrid retrieval** combines two approaches:

```
Query: "limitation of liability"
  ├── Semantic search: finds chunks about "capping damages" (similar meaning)
  └── Keyword search:  finds chunks containing exact "limitation of liability"
                ↓
        Merge and deduplicate → Unified result set
```

This gives us both **recall** (semantic finds related concepts) and
**precision** (keyword catches exact terms).

## Stack
- **LangChain** — orchestration and chains
- **ChromaDB** — vector store (semantic search)
- **BM25** — keyword-based search (via simple implementation)
- **Ollama** — local LLM + embeddings

## Step 1 — Imports

In [ ]:
# !pip install langchain langchain-ollama langchain-community chromadb -q

import re
from collections import Counter
import math

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain.schema import Document
from pydantic import BaseModel, Field
import json

print("All imports successful!")

## Step 2 — Sample Contract Documents

In [ ]:
contracts = {
    "SaaS Agreement": {
        "type": "SaaS",
        "content": """SOFTWARE AS A SERVICE AGREEMENT

1. DEFINITIONS
"Service" means the cloud-based software platform provided by Provider.
"Customer Data" means all data uploaded or created by Customer within the Service.
"Subscription Term" means the period specified in the Order Form.

2. SUBSCRIPTION AND ACCESS
2.1 Provider grants Customer a non-exclusive, non-transferable right to access
and use the Service during the Subscription Term, subject to the terms herein.
2.2 Customer shall not sublicense, reverse engineer, or create derivative works
based on the Service.

3. DATA OWNERSHIP AND PRIVACY
3.1 Customer retains all rights to Customer Data. Provider shall not access
Customer Data except to provide the Service or as required by law.
3.2 Provider will process Customer Data in accordance with its Privacy Policy
and applicable data protection laws.
3.3 Upon termination, Provider will make Customer Data available for export
for 30 days, after which it will be permanently deleted.

4. SERVICE LEVELS
4.1 Provider guarantees 99.9% uptime measured monthly, excluding scheduled
maintenance windows.
4.2 For each 0.1% below the guaranteed uptime, Customer receives a 5% credit
on the next month's invoice, up to a maximum of 30%.

5. LIMITATION OF LIABILITY
5.1 IN NO EVENT SHALL EITHER PARTY'S AGGREGATE LIABILITY EXCEED THE FEES PAID
BY CUSTOMER IN THE TWELVE (12) MONTHS PRECEDING THE CLAIM.
5.2 NEITHER PARTY SHALL BE LIABLE FOR INDIRECT, INCIDENTAL, SPECIAL,
CONSEQUENTIAL, OR PUNITIVE DAMAGES.
5.3 The foregoing limitations shall not apply to (a) breaches of confidentiality,
(b) IP infringement, or (c) willful misconduct.

6. TERMINATION
6.1 Either party may terminate for cause with 30 days written notice if the
other party materially breaches and fails to cure within that period.
6.2 Customer may terminate for convenience with 60 days written notice.
Upon convenience termination, no refund of prepaid fees shall be issued.
6.3 Provider may terminate immediately if Customer's use threatens the
security or performance of the Service for other customers.

7. INDEMNIFICATION
7.1 Provider shall indemnify Customer against third-party claims alleging
the Service infringes any patent, copyright, or trade secret.
7.2 Customer shall indemnify Provider against claims arising from Customer's
use of the Service or Customer Data."""
    },
    "NDA Agreement": {
        "type": "NDA",
        "content": """MUTUAL NON-DISCLOSURE AGREEMENT

1. DEFINITION OF CONFIDENTIAL INFORMATION
"Confidential Information" means any non-public information disclosed by
either party (Discloser) to the other (Recipient), whether orally, in writing,
or electronically, that is designated as confidential or that reasonably should
be understood to be confidential.

Exclusions: Information that (a) is or becomes publicly available through no
fault of Recipient, (b) was known to Recipient before disclosure, (c) is
independently developed by Recipient, or (d) is received from a third party
without restriction.

2. OBLIGATIONS
2.1 Recipient shall use Confidential Information solely for the Purpose.
2.2 Recipient shall protect Confidential Information with at least the same
degree of care used for its own confidential information, but no less than
reasonable care.
2.3 Recipient may disclose Confidential Information only to employees and
contractors with a need to know who are bound by similar obligations.

3. TERM
This Agreement remains in effect for 2 years from the Effective Date.
Obligations regarding Confidential Information survive for 3 years after
termination, except for trade secrets which are protected indefinitely.

4. RETURN OF MATERIALS
Upon termination or request, Recipient shall promptly return or destroy all
Confidential Information and certify destruction in writing.

5. REMEDIES
The parties acknowledge that breach may cause irreparable harm for which
monetary damages are inadequate. The Discloser shall be entitled to seek
injunctive relief without posting bond."""
    },
}

print(f"Loaded {len(contracts)} contracts")

## Step 3 — Build the Hybrid Retriever

We implement:
1. **Semantic retriever** — ChromaDB with embeddings
2. **Keyword retriever** — Simple BM25-style text search
3. **Hybrid merger** — Combines both result sets

In [ ]:
# Chunk contracts
all_docs = []
for name, data in contracts.items():
    doc = Document(
        page_content=data["content"],
        metadata={"contract": name, "type": data["type"]},
    )
    all_docs.append(doc)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=80,
    separators=["\n\n", "\n"],
)
chunks = splitter.split_documents(all_docs)
print(f"Split into {len(chunks)} chunks")

# Semantic retriever
embeddings = OllamaEmbeddings(model="nomic-embed-text-v2-moe")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="legal_clauses",
)
semantic_retriever = vectorstore.as_retriever(search_kwargs={"k": 6})

print(f"Semantic index: {vectorstore._collection.count()} vectors")

In [ ]:
# Simple keyword search (BM25-like)
def keyword_search(query: str, documents: list[Document], k: int = 6) -> list[Document]:
    """
    Simple keyword-based search using term frequency.
    
    In production, use a proper BM25 implementation
    (e.g., rank_bm25 library or Elasticsearch).
    """
    query_terms = set(re.findall(r'\w+', query.lower()))
    
    scored = []
    for doc in documents:
        doc_terms = re.findall(r'\w+', doc.page_content.lower())
        doc_counter = Counter(doc_terms)
        doc_len = len(doc_terms)
        
        # Simple TF scoring
        score = 0
        for term in query_terms:
            if term in doc_counter:
                tf = doc_counter[term] / doc_len
                score += tf
        
        scored.append((score, doc))
    
    scored.sort(key=lambda x: x[0], reverse=True)
    return [doc for _, doc in scored[:k] if scored[0][0] > 0]


# Hybrid retriever
def hybrid_retrieve(
    query: str,
    k: int = 5,
) -> list[Document]:
    """
    Combine semantic and keyword results, deduplicate.
    """
    # Get results from both
    semantic_results = semantic_retriever.invoke(query)
    keyword_results = keyword_search(query, chunks, k=6)
    
    # Merge and deduplicate by content
    seen = set()
    merged = []
    for doc in semantic_results + keyword_results:
        content_key = doc.page_content[:100]
        if content_key not in seen:
            seen.add(content_key)
            merged.append(doc)
    
    return merged[:k]


# Test hybrid search
results = hybrid_retrieve("limitation of liability")
print(f"Hybrid search returned {len(results)} results:")
for doc in results:
    print(f"  [{doc.metadata['contract']}] {doc.page_content[:80]}...")

## Step 4 — Clause Risk Analyzer

Use the LLM to assess whether a clause is favorable, neutral, or risky.

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0.1)

class ClauseRisk(BaseModel):
    clause_topic: str = Field(description="Topic of the clause (e.g., termination, liability)")
    risk_level: str = Field(description="low, medium, or high")
    summary: str = Field(description="One-sentence summary of what the clause says")
    risk_explanation: str = Field(description="Why this clause is risky or favorable")
    suggestion: str = Field(description="Suggested change to mitigate risk, or 'None' if acceptable")


risk_parser = JsonOutputParser(pydantic_object=ClauseRisk)

risk_prompt = ChatPromptTemplate.from_template(
    """You are a contract review attorney. Analyze this contract clause from the
perspective of the CUSTOMER (the party receiving the service or signing the agreement).

Assess:
- Is this clause standard or unusual?
- Does it favor the customer, the provider, or is it balanced?
- What risks does it pose to the customer?

{format_instructions}

Contract: {contract_name}
Clause:
---
{clause_text}
---

Risk assessment as JSON:"""
)

risk_chain = risk_prompt | llm | risk_parser

print("Risk analyzer ready!")

In [ ]:
# Analyze the termination clause
termination_docs = hybrid_retrieve("termination for convenience or cause", k=2)

print("⚖️ CLAUSE RISK ANALYSIS\n")
for doc in termination_docs:
    analysis = risk_chain.invoke({
        "contract_name": doc.metadata["contract"],
        "clause_text": doc.page_content,
        "format_instructions": risk_parser.get_format_instructions(),
    })
    
    level = analysis.get('risk_level', 'unknown').upper()
    icon = '🔴' if level == 'HIGH' else '🟡' if level == 'MEDIUM' else '🟢'
    print(f"{icon} [{level}] {analysis.get('clause_topic', '')} — {doc.metadata['contract']}")
    print(f"   Summary: {analysis.get('summary', '')}")
    print(f"   Risk: {analysis.get('risk_explanation', '')}")
    print(f"   Suggestion: {analysis.get('suggestion', '')}\n")

## Step 5 — Missing Clause Detection

Check a contract against a checklist of clauses that should be present.

In [ ]:
# Standard clauses expected in a SaaS agreement
EXPECTED_CLAUSES = [
    "Definitions",
    "Subscription and access rights",
    "Data ownership and privacy",
    "Service level agreement (SLA/uptime)",
    "Limitation of liability",
    "Termination clauses",
    "Indemnification",
    "Intellectual property rights",
    "Confidentiality",
    "Governing law and jurisdiction",
    "Force majeure",
    "Insurance requirements",
    "Audit rights",
    "Warranty",
]

missing_prompt = ChatPromptTemplate.from_template(
    """You are a contract review attorney. Given the following contract text,
determine if it contains a clause about: {clause_topic}

Answer with ONLY one of:
- FOUND: [brief description of what the clause says]
- MISSING: This clause is not present in the contract

Contract text:
{contract_text}

Assessment:"""
)

missing_chain = missing_prompt | llm | StrOutputParser()

# Check the SaaS agreement
contract_text = contracts["SaaS Agreement"]["content"]

print("📋 CLAUSE CHECKLIST — SaaS Agreement\n")
for clause in EXPECTED_CLAUSES:
    result = missing_chain.invoke({
        "clause_topic": clause,
        "contract_text": contract_text,
    })
    clean = result.strip().split("\n")[0]
    icon = "✅" if clean.upper().startswith("FOUND") else "❌"
    print(f"  {icon} {clause}: {clean[:80]}")

## Step 6 — Full Contract Q&A

In [ ]:
qa_prompt = ChatPromptTemplate.from_template(
    """You are a contract review assistant. Answer the question based on the
contract clauses provided. Be specific and cite clause numbers.

If a clause is unfavorable to the customer, flag it. If something important
is missing, mention it.

Contract clauses:
{context}

Question: {question}

Answer:"""
)


def ask_contract(question: str) -> None:
    """Ask a question about the contracts using hybrid retrieval."""
    print(f"❓ {question}")
    print("-" * 60)
    
    docs = hybrid_retrieve(question, k=4)
    context = "\n\n---\n\n".join(
        f"[{d.metadata['contract']}]\n{d.page_content}" for d in docs
    )
    
    chain = qa_prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})
    print(f"\n{answer}")
    print("=" * 60 + "\n")


ask_contract("What happens to my data if I cancel the SaaS subscription?")

In [ ]:
ask_contract("What is the maximum liability cap and what exceptions exist?")

In [ ]:
ask_contract("How long do confidentiality obligations last under the NDA?")

## 🧠 Key Concepts Recap

| Concept | What it does |
|---------|-------------|
| **Hybrid retrieval** | Combines semantic + keyword search for better recall |
| **Keyword search (BM25)** | Finds exact legal terminology matches |
| **Risk assessment** | LLM evaluates clauses for customer-unfriendly terms |
| **Missing clause detection** | Checks contracts against a standard checklist |
| **Structured output** | Pydantic models for consistent risk reports |
| **Clause-number citation** | References specific contract sections |

## 🔧 Customization Ideas

- **BM25 library**: Use `rank_bm25` for proper BM25 scoring
- **Contract comparison**: Compare two versions of the same contract
- **Industry templates**: Load different checklists for SaaS, employment, MSA
- **Redlining**: Suggest specific language changes for risky clauses